In [1]:
from random import randint
from math import sqrt
from torch import nn
from torch.nn import functional as F
import torch

In [2]:
def build_prompt(left, right, integer_width):
    a = f"{left:{integer_width}}"
    b = f"{right:{integer_width}}"
    c = f"{left + right:{integer_width + 1}}"

    return f"{a}+{b}={c}"

In [76]:
def write_dataset(integer_width):
    with open("dataset.txt", "w") as file:
        for _ in range(DATASET_SIZE):
            left = randint(0, MAX_INTEGER)
            right = randint(0, MAX_INTEGER)
            record = build_prompt(left, right, integer_width) + "\n"
            file.write(record)

In [77]:
def read_dataset(stoi):
    with open("dataset.txt", "r") as file:
        columns = len(file.readline()) - 1
        rows = sum(1 for _ in file) + 1

    dataset = torch.zeros(rows, columns)
    with open("dataset.txt", "r") as file:
        for y, row in enumerate(file):
            for x, item in enumerate(row[:-1]):
                dataset[y, x] = stoi[item]

    return dataset, dataset.size(1)

In [5]:
class Model(nn.Module):
    def __init__(self, block_size, n_embed, n_hidden, vocab_size):
        super().__init__()

        self.wte = nn.Embedding(vocab_size, n_embed)
        self.wpe = nn.Embedding(block_size, n_embed)
        self.decoder = Decoder(n_embed, n_hidden, vocab_size)
        self.output = nn.Linear(n_embed, vocab_size)

        self.position = torch.arange(block_size)  # block_size

    def forward(self, x, mask, is_causal):
        x = self.wte(x) + self.wpe(self.position)  # batch_size, block_size, n_embed
        x = self.decoder(x, mask, is_causal)  # batch_size, block_size, n_embed
        x = self.output(x)  # batch_size, block_size, vocab_size

        return x

    def parameter_count(self):
        return sum(parameter.numel() for parameter in self.parameters())

In [6]:
class Decoder(nn.Module):
    def __init__(self, n_embed, n_hidden, vocab_size):
        super().__init__()

        self.sa = SelfAttention(n_embed)
        self.ffn = FeedForwardNetwork(n_embed, n_hidden)

    def forward(self, x, mask, is_causal):
        x = x + self.sa(x, mask, is_causal)  # batch_size, block_size, n_embed
        x = x + self.ffn(x)  # batch_size, block_size, n_embed

        return x

In [7]:
class SelfAttention(nn.Module):
    def __init__(self, n_embed):
        super().__init__()

        self.q = nn.Linear(n_embed, n_embed)
        self.k = nn.Linear(n_embed, n_embed)
        self.v = nn.Linear(n_embed, n_embed)
        self.output = nn.Linear(n_embed, n_embed)

    def forward(self, x, mask, is_causal):
        q = self.q(x)  # batch_size, block_size, n_embed
        k = self.k(x)  # batch_size, block_size, n_embed
        v = self.v(x)  # batch_size, block_size, n_embed

        x = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, is_causal=is_causal)  # batch_size, block_size, n_embed
        x = self.output(x)  # batch_size, block_size, n_embed

        return x

In [8]:
class FeedForwardNetwork(nn.Module):
    def __init__(self, n_embed, n_hidden):
        super().__init__()
        
        self.input = nn.Linear(n_embed, n_hidden)
        self.relu = nn.ReLU()
        self.output = nn.Linear(n_hidden, n_embed)
        
    def forward(self, x):
        x = self.input(x)  # batch_size, block_size, n_hidden
        x = self.relu(x)  # batch_size, block_size, n_hidden
        x = self.output(x)  # batch_size, block_size, n_embed
        
        return x

In [9]:
BATCH_SIZE = 2**16
DATASET_SIZE = 2**24
LEARNING_RATE = 0.5
MAX_INTEGER = 9_999_999_999
N_EMBED = 2**4
N_HIDDEN = N_EMBED * 4
VOCABULARY = ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "+", "=", " "]

In [10]:
integer_width = len(str(MAX_INTEGER))
vocab_size = len(VOCABULARY)
stoi = {s: i for i, s in enumerate(VOCABULARY)}
itos = {i: s for i, s in enumerate(VOCABULARY)}

In [78]:
# write_dataset(integer_width)

In [79]:
dataset, block_size = read_dataset(stoi)

In [80]:
loss_fn = nn.CrossEntropyLoss()
model = Model(block_size, N_EMBED, N_HIDDEN, vocab_size)
model.parameter_count()

4173

In [81]:
%%time

for i in range(0, DATASET_SIZE - BATCH_SIZE, BATCH_SIZE):
    batch = dataset[i : i + BATCH_SIZE]  # batch_size, block_size
    predictions = model(batch, None, True)[:, :-1, :].transpose(1, 2)  # batch_size, vocab_size, block_size-1
    targets = batch[:, 1:]  # batch_size, block_size-1

    loss = loss_fn(predictions, targets)
    loss.backward()

    if i / BATCH_SIZE % 10 == 0:
        print(loss.item())

    with torch.no_grad():
        for parameter in model.parameters():
            parameter -= LEARNING_RATE * parameter.grad
            parameter.grad.zero_()

CPU times: user 2.53 ms, sys: 981 μs, total: 3.51 ms
Wall time: 8.85 ms


RuntimeError: Expected tensor for argument #1 'indices' to have one of the following scalar types: Long, Int; but got torch.FloatTensor instead (while checking arguments for embedding)

In [ ]:
x = build_prompt(1111111111, 1111111111, integer_width)
i = x.find("=") + 1

mask = torch.tensor([True] * i + [False] * (len(x) - i))
string = torch.tensor([stoi[s] if ii<i else stoi[" "] for ii,s in enumerate(x)])

with torch.no_grad():
    while i < len(mask):

        
        
        logits = model(string, mask, False)       
        logits = logits[i-1]
        logits = logits.softmax(0)
        logits = torch.multinomial(logits,1,replacement=True).item()
        
        

        string[i] = logits
        mask[i] = True

        if logits == stoi["."]:
            break
        
        i += 1

"".join([itos[i.item()] for i in string])